# HDB resale flat prices — ETL pipeline

This notebook is Part 1. Window: **January 2012 – December 2016**.

Run **all cells top to bottom**. I wrote the judgements in `docs/ASSUMPTIONS.md`. Part 2 (Tableau / Athena) is in `architecture/`. That is the design, not another table.

In [1]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 88)
pd.set_option("display.width", 140)

# Notebook may be launched from /notebooks or the repo root.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from hdb_etl.config import MONTH_END, MONTH_START, REFERENCE_MONTH, RESALE_IDENTIFIER_COL
from hdb_etl.extract import ensure_raw_files, fetch_collection_datasets, raw_inventory
from hdb_etl.combine import combine_master, load_raw_frames
from hdb_etl.profile import domain_comparison, profile_dataset
from hdb_etl.validate import apply_validation, build_jan2012_reference
from hdb_etl.transform import (
    add_hashes,
    add_remaining_lease,
    add_resale_identifier,
    build_resale_identifier,
    format_remaining_lease,
    hashed_zone,
    remaining_lease_parts,
)
from hdb_etl.pipeline import run_pipeline

print(f"Project root: {ROOT}")
print(f"Window: {MONTH_START} → {MONTH_END}; Jan 2012 is the authoritative month")

Project root: /Users/benjaminng/Desktop/HDB technical assessment
Window: 2012-01 → 2016-12; Jan 2012 is the authoritative month


## 1. Extract (files as-is)

I read collection 189 metadata from the public API. Raw CSVs come from `data/raw/` or get streamed from data.gov.sg if that folder is empty. I do not edit them.

In [2]:
# Live metadata call — proves extract is not a hardcoded filename list.
try:
    datasets = fetch_collection_datasets()
    display(pd.DataFrame(datasets)[["datasetId", "name", "coverageStart", "coverageEnd"]])
except Exception as exc:
    print(f"Metadata API unavailable this run ({exc}); continuing with local raw files.")

raw_files = ensure_raw_files(ROOT)
print("Raw zone (as-is):")
for item in raw_inventory(raw_files):
    print(f"  {item['file']}  ({item['bytes']:,} bytes)")

,datasetId,name,coverageStart,coverageEnd
0,d_8b84c4ee58e3cfc0ece0d773c8ca6abc,Resale flat prices based on registration date from Jan-2017 onwards,2017-01-01T08:00:00+08:00,2026-09-01T08:00:00+08:00
1,d_43f493c6c50d54243cc1eab0df142d6a,"Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012",2000-01-01T08:00:00+08:00,2012-02-01T08:00:00+08:00
2,d_2d5ff9ea31397b66239f245f57751537,"Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014",2012-03-01T08:00:00+08:00,2014-12-01T08:00:00+08:00
3,d_ebc5ab87086db484f88045b47411ebc5,"Resale Flat Prices (Based on Approval Date), 1990 - 1999",1990-01-01T08:00:00+08:00,1999-12-01T08:00:00+08:00
4,d_ea9ed51da2787afaf8e51f827c304208,"Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016",2015-01-01T08:00:00+08:00,2016-12-01T08:00:00+08:00


Raw zone (as-is):
  Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv  (22,635,623 bytes)
  Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv  (29,369,945 bytes)
  Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv  (3,070,924 bytes)
  Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv  (4,160,771 bytes)
  Resale flat prices based on registration date from Jan-2017 onwards.csv  (23,770,853 bytes)


## 2. Combine + profile

I union every column from every file (`remaining_lease` included, null before 2015), then keep rows with `month` in 2012-01 … 2016-12.

In [3]:
frames = load_raw_frames(raw_files)
schema_rows = []
for name, frame in frames.items():
    schema_rows.append(
        {
            "source_file": name,
            "rows": len(frame),
            "has_remaining_lease": "remaining_lease" in frame.columns,
            "month_min": frame["month"].min(),
            "month_max": frame["month"].max(),
        }
    )
display(pd.DataFrame(schema_rows))

master = combine_master(frames)  # filter happens here, raw files are untouched
print(f"Master rows in window: {len(master):,}  columns: {list(master.columns)}")
report = profile_dataset(master, frames)
print(f"Months: {report['nunique_months']}  {report['month_min']} → {report['month_max']}")
display(pd.DataFrame(report["source_counts"]))

,source_file,rows,has_remaining_lease,month_min,month_max
0,"Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv",287196,False,1990-01,1999-12
1,"Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv",369651,False,2000-01,2012-02
2,"Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv",37153,True,2015-01,2016-12
3,"Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv",52203,False,2012-03,2014-12
4,Resale flat prices based on registration date from Jan-2017 onwards.csv,240345,True,2017-01,2026-09


Master rows in window: 92,544  columns: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price', 'source_file']
Months: 60  2012-01 → 2016-12


,source_file,rows
0,"Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv",52203
1,"Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv",37153
2,"Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv",3188


## 3. Validate against January 2012 (authoritative set)

I treat town, flat type, flat model, and `storey_range` as **closed sets** from Jan 2012. Date uses that month’s `YYYY-MM` format plus the assigned window. Failures leave Cleaned. See `docs/ASSUMPTIONS.md` — this is why most 2012–2014 5-storey bands get quarantined.

In [4]:
reference = build_jan2012_reference(master)
print("Jan 2012 towns:", sorted(reference["town"]))
print("Jan 2012 flat types:", sorted(reference["flat_type"]))
print("Jan 2012 flat models:", sorted(reference["flat_model"]))
print("Jan 2012 storey_range:", sorted(reference["storey_range"]))
display(domain_comparison(master, {k: v for k, v in reference.items() if k != "month_format"}))

validated = apply_validation(master, reference)
print("Structural fails:", int(validated["structural_fail"].sum()))

Jan 2012 towns: ['ANG MO KIO', 'BEDOK', 'BISHAN', 'BUKIT BATOK', 'BUKIT MERAH', 'BUKIT PANJANG', 'BUKIT TIMAH', 'CENTRAL AREA', 'CHOA CHU KANG', 'CLEMENTI', 'GEYLANG', 'HOUGANG', 'JURONG EAST', 'JURONG WEST', 'KALLANG/WHAMPOA', 'MARINE PARADE', 'PASIR RIS', 'PUNGGOL', 'QUEENSTOWN', 'SEMBAWANG', 'SENGKANG', 'SERANGOON', 'TAMPINES', 'TOA PAYOH', 'WOODLANDS', 'YISHUN']
Jan 2012 flat types: ['1 ROOM', '2 ROOM', '3 ROOM', '4 ROOM', '5 ROOM', 'EXECUTIVE', 'MULTI-GENERATION']
Jan 2012 flat models: ['Adjoined flat', 'Apartment', 'Improved', 'Maisonette', 'Model A', 'Model A-Maisonette', 'Model A2', 'Multi Generation', 'New Generation', 'Premium Apartment', 'Simplified', 'Standard', 'Terrace']
Jan 2012 storey_range: ['01 TO 03', '04 TO 06', '07 TO 09', '10 TO 12', '13 TO 15', '16 TO 18', '19 TO 21', '22 TO 24', '25 TO 27', '28 TO 30', '31 TO 33', '34 TO 36']


,field,reference_count,observed_count,new_values,new_value_count,reference_only
0,town,26,26,,0,
1,flat_type,7,7,,0,
2,flat_model,13,20,"2-room, DBSS, Improved-Maisonette, Premium Apartment Loft, Premium Maisonette, Type ...",7,
3,storey_range,12,25,"01 TO 05, 06 TO 10, 11 TO 15, 16 TO 20, 21 TO 25, 26 TO 30, 31 TO 35, 36 TO 40, 37 T...",13,


Structural fails: 7411


## 4. Remaining lease, dedup, anomalies, identifier, hash

`run_pipeline` does the rest and writes the five zones. I keep anomalies **out** of Cleaned. The column name is `Resale Identifier`.

In [5]:
# Worked examples before the full run (brief: 99-year lease; block 19 → 019).
for commence, month in [("1986", "2015-01"), ("1986", "2015-06"), ("1977", "2012-01")]:
    years, months = remaining_lease_parts(commence, month)
    print(f"commence {commence}, {month} → {format_remaining_lease(years, months)}")

print("Identifier spec example:", build_resale_identifier("19", 230000, "2012-01", "ANG MO KIO"))
print("Block 6A:", build_resale_identifier("6A", 230000, "2012-01", "ANG MO KIO"))

result = run_pipeline(root=ROOT, persist=True, drop_anomalies=True)
print(f"Cleaned:     {len(result['cleaned']):,}")
print(f"Quarantined: {len(result['quarantined']):,}")
print(f"Transformed: {len(result['transformed']):,}")
print(f"Hashed:      {len(result['hashed']):,}")
print("Identifier uniqueness:", result["profile_report"]["identifier_uniqueness"])

commence 1986, 2015-01 → 70 years 00 months
commence 1986, 2015-06 → 69 years 07 months
commence 1977, 2012-01 → 64 years 00 months
Identifier spec example: S0192301A
Block 6A: S0062301A


Cleaned:     76,959
Quarantined: 15,585
Transformed: 76,959
Hashed:      76,959
Identifier uniqueness: {'cleaned_rows': 76959, 'distinct_resale_identifier': 66258, 'identifier_collisions': 10701, 'distinct_hashed_record_id': 76959}


## 5. Inspect zones

I expect quarantine reasons for `unknown_storey_range` / `unknown_flat_model` (Jan 2012 closed set), duplicates, and anomalies. The hashed zone should not have the plaintext identifier.

In [6]:
q = result["quarantined"]
if not q.empty:
    display(q["quarantine_reason"].value_counts().rename("rows").to_frame())

display(
    result["transformed"][
        ["month", "town", "flat_type", "block", "avg_resale_price_group", RESALE_IDENTIFIER_COL, "resale_price"]
    ].head(8)
)

assert RESALE_IDENTIFIER_COL not in result["hashed"].columns
display(result["hashed"][["month", "town", "block", "hashed_identifier", "hashed_record_id"]].head(5))

print("Written:")
for zone, payload in result["written"].items():
    print(f"  {zone}: {payload}")

,rows
quarantine_reason,
unknown_storey_range,6919
area_anomaly,4990
price_anomaly,1430
duplicate_lower_price,1119
unknown_flat_model,436
price_anomaly;area_anomaly,371
duplicate_tie,264
unknown_storey_range;unknown_flat_model,56


,month,town,flat_type,block,avg_resale_price_group,Resale Identifier,resale_price
0,2012-01,ANG MO KIO,3 ROOM,118,348208.024444,S1183401A,320000.0
1,2012-01,ANG MO KIO,3 ROOM,121,348208.024444,S1213401A,382800.0
2,2012-01,ANG MO KIO,4 ROOM,126,474485.818182,S1264701A,452000.0
3,2012-01,ANG MO KIO,3 ROOM,151,348208.024444,S1513401A,302000.0
4,2012-01,ANG MO KIO,3 ROOM,154,348208.024444,S1543401A,321000.0
5,2012-01,ANG MO KIO,3 ROOM,157,348208.024444,S1573401A,297000.0
6,2012-01,ANG MO KIO,3 ROOM,163,348208.024444,S1633401A,340000.0
7,2012-01,ANG MO KIO,2 ROOM,170,263160.000000,S1702601A,260000.0


,month,town,block,hashed_identifier,hashed_record_id
0,2012-01,ANG MO KIO,118,3e7d1629955bf8874d7b98e4bbd2aca86efb68c548d00476b4ab011e7d8c2129,4bc135395c403ee7c15aa433eac58e25735524bb06d9671d299912b601d209f8
1,2012-01,ANG MO KIO,121,ba7731ca106ec514d5471dfa92bd053d5c53b02d097b6b0afb2a3cc9a6e11b18,5626f86de4044dba9fdfddce52aac80097a2bae2705db7a779ca9bb450ab75db
2,2012-01,ANG MO KIO,126,02e68094e7e2df51ad56ffac2c6f3f0adfd18236661dbfa287a508fdde4c57f1,477eb1569d9a011da6c7478de0e748d068f0b2dbfe34f80b8ace4487f57c282d
3,2012-01,ANG MO KIO,151,10882c6d534f79a06a8cbac9c62de905386e109232634c384937c2cd1cfae335,172ed3611aa06659b44cabd433d2f757506269c5e65a46b86c7a5009ee64ba33
4,2012-01,ANG MO KIO,154,8dd77c00a6f9e876c0fc5fb1047a31c1d5de2a6e2bb3aca5f626a3a535fe7db8,22094215378d51125cdc37b4e6fa0e7b95d8e34b390af1e8872dfa2dda690529


Written:
  cleaned: {'csv': PosixPath('/Users/benjaminng/Desktop/HDB technical assessment/data/cleaned/cleaned.csv'), 'parquet': PosixPath('/Users/benjaminng/Desktop/HDB technical assessment/data/cleaned/cleaned.parquet')}
  transformed: {'csv': PosixPath('/Users/benjaminng/Desktop/HDB technical assessment/data/transformed/transformed.csv'), 'parquet': PosixPath('/Users/benjaminng/Desktop/HDB technical assessment/data/transformed/transformed.parquet')}
  hashed: {'csv': PosixPath('/Users/benjaminng/Desktop/HDB technical assessment/data/hashed/hashed.csv'), 'parquet': PosixPath('/Users/benjaminng/Desktop/HDB technical assessment/data/hashed/hashed.parquet')}
  quarantined: {'csv': PosixPath('/Users/benjaminng/Desktop/HDB technical assessment/data/quarantined/quarantined.csv'), 'parquet': PosixPath('/Users/benjaminng/Desktop/HDB technical assessment/data/quarantined/quarantined.parquet')}
  domain_drift: {'csv': PosixPath('/Users/benjaminng/Desktop/HDB technical assessment/data/quarantin

## 6. Part 2 — architecture (not a data table)

Tableau + Athena, private traffic, PNG/draw.io, public git, and leaving the assignment PDF out are in this folder, not in another CSV:

- `architecture/01-data-ingestion.png` — public data.gov.sg → private VPC → S3
- `architecture/02-data-exploitation.png` — Tableau in a **second** private VPC → Athena JDBC over Interface/Gateway endpoints
- `architecture/ARCHITECTURE.md` — CIDRs, IAM, JDBC fields, and my assumptions

Open the PNGs in the repo. I did not commit `Data Engineering Technical Test.pdf`.